# XLM-R-base — SecBangla `splits_v2` benchmark (unadapted baseline, 3-class)

Fine-tunes `FacebookAI/xlm-roberta-base` for the 3-class SecBangla decision task (`ALLOW` / `REVIEW` / `BLOCK`).

This is the **unadapted XLM-R Base baseline** of the BanglaBlocker paper. It replaces the earlier binary-objective XLM-R Base run and uses exactly the same protocol as the XLM-R Large baseline (`Finetuning-xlm-r-large.ipynb`): plain `AutoModelForSequenceClassification` head, no TAPT/TLM, no auxiliary heads, pooled language forms, seeds 42–44. The only differences are the backbone and the batch/learning-rate settings for a Base-size model. Because it saves per-prompt 3-class predictions, every metric in the paper can be computed for it: three-class Macro-F1 and accuracy, detection recall, intent-level F1, bootstrap confidence intervals, and LODO.

**Dataset:** cleaned `splits_v2` release:
- `train.csv`: 2,850 scenarios
- `val.csv`: 450 scenarios
- `heldout_test.csv`: 900 scenarios
- `decision_matched_test.csv`: 270-scenario balanced diagnostic subset

**Training protocol:** one pooled multilingual model per experiment/seed. Each source scenario contributes Bangla, Banglish, code-mix, and English realizations. Evaluation reports the pooled result plus per-text-variant slices.

**Kaggle:** attach the extracted `splits_v2` dataset (or `splits_v2.zip`), enable a GPU (T4 is enough), and choose *Run all*. With the defaults below it trains 12 models (main + 3 leave-one-domain-out experiments × seeds 42, 43, 44). Optional: set `SMOKE_RUN = True` first for a few-minute pipeline check.

**When it finishes,** download `xlmr_base_splits_v2_outputs.zip` from `/kaggle/working/` (created by the last cell) and send it back. The files needed for the paper are in `predictions/` (`main_all_heldout_test_seed4*.csv` and the `lodo_*` files) plus `run_meta.json` and `metrics_long.csv`.


In [ ]:
# ============================================================
# CELL 1 — SETUP + CONFIGURATION
# ============================================================
!pip -q install --upgrade "transformers>=4.44,<5" "scikit-learn>=1.3" "sentencepiece>=0.1.99" "protobuf" 2>/dev/null || echo "pip install skipped (offline)"

import os, sys, subprocess, json

# ============================================================
# RUN CONTROLS
# ============================================================

# Full run by default. Set True for an optional few-minute pipeline check.
SMOKE_RUN = False

# False = run only TRAIN_DOMAINS/TEST_DOMAIN below
# True  = run main + all 3 leave-one-domain-out (LODO) experiments
RUN_FULL_EXPERIMENTS = True

TEXT_VARIANTS = [
    "bangla",
    "banglish",
    "code_mix",
    "english",
]

SEEDS = [42, 43, 44]

FIGURE_SEED = SEEDS[0]
MIN_SLICE_SUPPORT_OVERRIDE = None
FORCE_RETRAIN = False

# Checkpoints are not needed for the paper. Keep False for benchmarking unless
# you explicitly need deployable checkpoints.
SAVE_CHECKPOINTS = False

# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MODEL_SLUG = "xlmr_base_splits_v2"

# ============================================================
# EXPERIMENT CONFIG  (used only when RUN_FULL_EXPERIMENTS=False)
# ============================================================

TRAIN_DOMAINS = [
    "banking",
    "healthcare",
    "e-commerce",
]

TEST_DOMAIN = None

# ============================================================
# TRAINING
# ============================================================

AUX_HEADS = False
MAX_LEN = 128
EPOCHS = 10
LR = 2e-5   # Base-size default (XLM-R Large baseline uses 1e-5)

# XLM-R Base fits a batch of 16 on a Kaggle T4. GRAD_ACCUM keeps the same
# effective batch size of 32 examples as the XLM-R Large baseline.
BATCH = 16
GRAD_ACCUM = 2

WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
GRADIENT_CHECKPOINTING = False   # not needed for Base; faster
USE_MIXED_PRECISION = True
EVAL_BATCH = 64

PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3

# ============================================================
# SMOKE OVERRIDES
# ============================================================

SMOKE_TRAIN_N = 64
SMOKE_VAL_N = 32
SMOKE_TEST_N = 32
SMOKE_MAX_STEPS = 10
SMOKE_EPOCHS = 1
SMOKE_SEEDS = [42]

# Optional local sharding. Kaggle normally leaves these at 0/1.
SHARD_INDEX = int(os.environ.get("SHARD_INDEX", 0))
SHARD_COUNT = int(os.environ.get("SHARD_COUNT", 1))

print("MODEL_NAME  :", MODEL_NAME)
print("MODEL_SLUG  :", MODEL_SLUG)
print("SMOKE_RUN   :", SMOKE_RUN)
print("FULL EXPS   :", RUN_FULL_EXPERIMENTS)
print("SEEDS       :", SMOKE_SEEDS if SMOKE_RUN else SEEDS)
print("SHARD       :", SHARD_INDEX, "/", SHARD_COUNT)


## Cell 2 — Common benchmark block

Shared data, leakage-control, metrics, bootstrap, artifact, and Kaggle path utilities for the cleaned `splits_v2` protocol.


In [ ]:
# ===== COMMON BLOCK — splits_v2 benchmark utilities =====
COMMON_VERSION = "2.1.0-splits_v2"

import os
import gc
import json
import math
import time
import random
import platform
import warnings
import zipfile
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    log_loss,
)

# ------------------------------------------------------------------
# Labels
# ------------------------------------------------------------------
LABELS = ["ALLOW", "REVIEW", "BLOCK"]
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
ID2LABEL = {i: lab for lab, i in LABEL2ID.items()}
N_LABELS = len(LABELS)

# ------------------------------------------------------------------
# Contracts
# ------------------------------------------------------------------
TEXT_VARIANTS_ALL = ["bangla", "banglish", "code_mix", "english"]
EVAL_SETS = ["val", "heldout_test", "decision_matched"]

# Columns that must never reach the tokenizer. They are labels, rationales,
# or metadata/annotations that can be strongly label-correlated.
FORBIDDEN_INPUT_COLS = [
    "note",
    "decision",
    "security_intent",
    "risk_level",
    "hard_negative_type",
]

SLICE_COLS = ["domain", "security_intent", "risk_level", "hard_negative_type", "in_decision_matched_test"]

# ------------------------------------------------------------------
# Unified (pooled-variant) training.
#   Each model trains on ALL four text variants at once. melt_variants()
#   stacks the four variant columns into POOLED_COL (long form) with a
#   text_variant tag, so a source row becomes 4 rows. POOLED_TAG is the
#   run-key/filename slot that replaces the old per-variant loop.
# ------------------------------------------------------------------
POOLED_COL = "text"
POOLED_TAG = "all"

MIN_SLICE_SUPPORT = 20
N_BOOTSTRAP = 1000
BOOTSTRAP_METRICS = ["f1_macro", "accuracy"]
N_CALIB_BINS = 15

MASTER_FILENAME = "SecBangla_clean_master_with_splits.csv"
DERIVED_FILES = {"train": "train.csv", "val": "val.csv", "heldout": "heldout_test.csv", "matched": "decision_matched_test.csv"}

EXPERIMENTS = OrderedDict([
    ("main", {
        "train_domains": ["banking", "healthcare", "e-commerce"],
        "test_domain": None,
    }),
    ("lodo_banking", {
        "train_domains": ["healthcare", "e-commerce"],
        "test_domain": "banking",
    }),
    ("lodo_healthcare", {
        "train_domains": ["banking", "e-commerce"],
        "test_domain": "healthcare",
    }),
    ("lodo_ecommerce", {
        "train_domains": ["banking", "healthcare"],
        "test_domain": "e-commerce",
    }),
])

EXPECTED_EXPERIMENTS = set(EXPERIMENTS)
EXPECTED_VARIANTS = set(TEXT_VARIANTS_ALL)
EXPECTED_EVAL_SETS = set(EVAL_SETS)

PRED_COLUMNS = [
    "model_slug",
    "experiment",
    "seed",
    "eval_set",
    "text_variant",
    "source_text_variant",
    "id",
    "scenario_family_id",
    "split",
    "in_decision_matched_test",
    "y_true",
    "y_pred",
    "p_ALLOW",
    "p_REVIEW",
    "p_BLOCK",
    "domain",
    "security_intent",
    "risk_level",
    "hard_negative_type",
]

METRICS_LONG_COLUMNS = [
    "model_slug",
    "experiment",
    "seed",
    "eval_set",
    "text_variant",
    "slice_type",
    "slice_value",
    "metric",
    "value",
    "n_support",
    "low_support",
]

# Exact metric key set emitted for slice_type == "overall".
METRIC_KEYS = [
    # classification
    "accuracy", "balanced_accuracy",
    "f1_macro", "f1_micro", "f1_weighted",
    "precision_macro", "precision_weighted",
    "recall_macro", "recall_weighted",
    "cohen_kappa", "mcc",
    # per class
    "precision_ALLOW", "recall_ALLOW", "f1_ALLOW", "support_ALLOW",
    "precision_REVIEW", "recall_REVIEW", "f1_REVIEW", "support_REVIEW",
    "precision_BLOCK", "recall_BLOCK", "f1_BLOCK", "support_BLOCK",
    # binary collapse: ALLOW vs (REVIEW + BLOCK)
    "bin_auprc", "bin_roc_auc",
    "bin_fpr_at_tpr95", "bin_fpr_at_tpr99",
    "bin_tpr_at_fpr1", "bin_tpr_at_fpr5",
    "bin_precision_argmax", "bin_recall_argmax", "bin_f1_argmax",
    "bin_precision_tuned", "bin_recall_tuned", "bin_f1_tuned",
    "bin_best_f1_threshold",
    # calibration
    "ece", "mce", "brier_multiclass", "brier_binary", "nll",
    "temperature", "ece_before_temp", "ece_after_temp",
    # safety
    "over_refusal_rate", "over_refusal_block_rate", "over_refusal_n",
    "over_escalation_rate", "over_escalation_n",
    "attack_recall", "attack_n",
    "catastrophic_miss_rate", "block_to_review_rate", "block_n",
    "review_to_allow_rate", "review_to_block_rate", "review_n",
    # uncertainty
    "f1_macro_ci_low", "f1_macro_ci_high",
    "accuracy_ci_low", "accuracy_ci_high",
]

# ------------------------------------------------------------------
# Path resolution
# ------------------------------------------------------------------
REQUIRED_DATA_FILES = [
    MASTER_FILENAME,
    DERIVED_FILES["train"],
    DERIVED_FILES["val"],
    DERIVED_FILES["heldout"],
    DERIVED_FILES["matched"],
]

DATA_DIR_CANDIDATES = [
    # Common Kaggle layouts
    "/kaggle/input/splits-v2/splits_v2",
    "/kaggle/input/sec-bangla-guard-v2/splits_v2",
    "/kaggle/input/sec-bangla-guard/splits_v2",
    "/kaggle/input/splits_v2",
    # Local / uploaded archive extraction
    "./splits_v2",
    "../splits_v2",
    "/mnt/data/splits_v2",
]


def _is_valid_data_dir(path):
    return os.path.isdir(path) and all(
        os.path.exists(os.path.join(path, name)) for name in REQUIRED_DATA_FILES
    )


def resolve_data_dir(candidates=None, verbose=True):
    """Find the cleaned splits_v2 directory.

    The function supports both an extracted Kaggle Dataset and a raw
    `splits_v2.zip` attached as an input dataset.
    """
    candidates = list(candidates or DATA_DIR_CANDIDATES)

    if os.path.isdir("/kaggle/input"):
        # First prefer already-extracted CSVs.
        for root, dirs, files in os.walk("/kaggle/input"):
            if MASTER_FILENAME in files:
                candidates.append(root)

    seen = set()
    for cand in candidates:
        cand = os.path.abspath(cand)
        if cand in seen:
            continue
        seen.add(cand)
        if _is_valid_data_dir(cand):
            if verbose:
                print("resolved DATA_DIR :", cand)
                for key, fname in DERIVED_FILES.items():
                    path = os.path.join(cand, fname)
                    print("  {:8s} {:36s} {}".format(
                        key, fname, "ok" if os.path.exists(path) else "MISSING"))
            return cand

    # Kaggle fallback: inspect attached ZIP files and extract the one that
    # actually contains the splits_v2 master CSV.
    if os.path.isdir("/kaggle/input"):
        zip_candidates = []
        for root, dirs, files in os.walk("/kaggle/input"):
            for fname in files:
                if fname.lower().endswith(".zip"):
                    zip_candidates.append(os.path.join(root, fname))

        for zpath in zip_candidates:
            try:
                with zipfile.ZipFile(zpath, "r") as zf:
                    names = zf.namelist()
                    if not any(
                        n == MASTER_FILENAME or n.endswith("/" + MASTER_FILENAME)
                        for n in names
                    ):
                        continue
                    extract_root = "/kaggle/working/_secban_v2_data"
                    os.makedirs(extract_root, exist_ok=True)
                    zf.extractall(extract_root)
            except zipfile.BadZipFile:
                continue

            # The release normally contains a top-level splits_v2/ directory.
            for cand in [
                os.path.join(extract_root, "splits_v2"),
                extract_root,
            ]:
                if _is_valid_data_dir(cand):
                    if verbose:
                        print("extracted dataset:", zpath)
                        print("resolved DATA_DIR :", cand)
                    return cand

    raise FileNotFoundError(
        "Could not find the cleaned splits_v2 dataset. Expected these files "
        "in one directory:\n  " + "\n  ".join(REQUIRED_DATA_FILES) +
        "\n\nKaggle: attach the extracted dataset or a file named splits_v2.zip, "
        "then rerun this cell."
    )


def resolve_out_dir(model_slug, verbose=True):
    """Kaggle writes to /kaggle/working, a local box writes to ./outputs."""
    root = "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "./outputs"
    out_dir = os.path.join(root, model_slug)
    os.makedirs(out_dir, exist_ok=True)
    for sub in ["predictions", "curves", "figures", "tables", "training"]:
        os.makedirs(os.path.join(out_dir, sub), exist_ok=True)
    if verbose:
        print("resolved OUT_DIR  :", out_dir)
    return out_dir


def resolve_device_profile(verbose=True):
    """Pick dtype / attention implementation from compute capability.

    T4 is Turing (sm_75) and has no bf16 tensor cores.
    RTX 4080 is Ada (sm_89) and does.
    """
    profile = {
        "platform": "kaggle" if os.path.isdir("/kaggle/working") else "local",
        "cuda_available": torch.cuda.is_available(),
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "gpu_name": None,
        "capability": None,
        "is_ampere_plus": False,
        "dtype": "float32",
        "bf16": False,
        "fp16": False,
        "attn_impl": "eager",
        "tf32": False,
    }

    if not torch.cuda.is_available():
        if verbose:
            print("device profile    : CPU only")
            print(json.dumps(profile, indent=2))
        return profile

    cap = torch.cuda.get_device_capability(0)
    profile["gpu_name"] = torch.cuda.get_device_name(0)
    profile["capability"] = "{}.{}".format(cap[0], cap[1])
    profile["is_ampere_plus"] = cap >= (8, 0)

    if profile["is_ampere_plus"]:
        profile["dtype"] = "bfloat16"
        profile["bf16"] = True
        profile["attn_impl"] = "sdpa"
        profile["tf32"] = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    else:
        profile["dtype"] = "float16"
        profile["fp16"] = True
        profile["attn_impl"] = "eager"

    if verbose:
        print("device profile    :")
        for k, v in profile.items():
            print("  {:16s} {}".format(k, v))
    return profile


def torch_dtype_of(profile):
    return {"bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32}[profile["dtype"]]


def is_kaggle_t4(profile):
    return profile["platform"] == "kaggle" and not profile["is_ampere_plus"]


# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


# ------------------------------------------------------------------
# Data
# ------------------------------------------------------------------
def duplicate_ids(master, verbose=True):
    """Audit helper retained for reporting only.

    splits_v2 is already cleaned/quarantined before release, so no rows are
    removed inside the training notebook. We nevertheless verify that the
    released train/val/heldout partitions have no exact cross-split text leak.
    """
    split_ids, dom_ids = set(), set()
    per_variant = {}
    for col in TEXT_VARIANTS_ALL:
        n_split = master.groupby(col)["split"].nunique()
        dup_split = set(n_split[n_split > 1].index)
        split_ids |= set(master.loc[master[col].isin(dup_split), "id"])

        n_dom = master.groupby(col)["domain"].nunique()
        dup_dom = set(n_dom[n_dom > 1].index)
        dom_ids |= set(master.loc[master[col].isin(dup_dom), "id"])
        per_variant[col] = {
            "cross_split_texts": len(dup_split),
            "cross_domain_texts": len(dup_dom),
        }

    if verbose:
        print("released-text audit:", per_variant)
        print("  ids touching cross-split duplicate text:", len(split_ids))
        print("  ids touching cross-domain duplicate text:", len(dom_ids))
    return sorted(split_ids | dom_ids), per_variant


def _assert_family_disjoint(a, b, a_name, b_name):
    overlap = set(a["scenario_family_id"]) & set(b["scenario_family_id"])
    assert not overlap, (
        "scenario-family leakage between {} and {}: {} families".format(
            a_name, b_name, len(overlap))
    )


def load_splits(train_domains, test_domain, master_csv, verbose=False):
    """Load the cleaned v2 split protocol.

    Main experiment:
      train -> all domains / split=train
      val -> all domains / split=val
      heldout_test -> all 900 held-out scenarios
      decision_matched -> the 270-scenario balanced diagnostic subset

    LODO experiment:
      train/val -> only the two source domains
      heldout_test -> 300 held-out scenarios from the unseen target domain
      decision_matched -> 90 balanced target-domain diagnostic scenarios

    The target domain is never used for LODO training, early stopping,
    temperature scaling, or threshold tuning.
    """
    master = pd.read_csv(master_csv)

    required = [
        "id", "domain", "split", "decision", "scenario_family_id",
        "in_decision_matched_test",
    ] + TEXT_VARIANTS_ALL + [
        "security_intent", "risk_level", "hard_negative_type"
    ]
    missing = [c for c in required if c not in master.columns]
    assert not missing, "missing columns: {}".format(missing)
    assert master["id"].is_unique, "id column must be unique"
    assert set(master["split"]) <= {"train", "val", "heldout_test"}
    assert set(master["decision"]) <= set(LABELS)

    for col in TEXT_VARIANTS_ALL:
        assert master[col].notna().all(), "nulls in text variant {}".format(col)

    train = master[(master.domain.isin(train_domains)) & (master.split == "train")]
    val = master[(master.domain.isin(train_domains)) & (master.split == "val")]

    if test_domain is None:
        heldout = master[master.split == "heldout_test"]
    else:
        heldout = master[
            (master.domain == test_domain) &
            (master.split == "heldout_test")
        ]

    matched = heldout[heldout["in_decision_matched_test"].astype(bool)]

    out = {
        "train": train.reset_index(drop=True),
        "val": val.reset_index(drop=True),
        "heldout_test": heldout.reset_index(drop=True),
        "decision_matched": matched.reset_index(drop=True),
    }

    # ID and semantic-family disjointness.
    assert set(out["train"].id).isdisjoint(set(out["val"].id))
    assert set(out["train"].id).isdisjoint(set(out["heldout_test"].id))
    assert set(out["val"].id).isdisjoint(set(out["heldout_test"].id))
    assert set(out["decision_matched"].id) <= set(out["heldout_test"].id)

    _assert_family_disjoint(out["train"], out["val"], "train", "val")
    _assert_family_disjoint(out["train"], out["heldout_test"], "train", "heldout_test")
    _assert_family_disjoint(out["val"], out["heldout_test"], "val", "heldout_test")

    if test_domain is not None:
        assert test_domain not in set(out["train"].domain)
        assert test_domain not in set(out["val"].domain)
        assert set(out["heldout_test"].domain) == {test_domain}

    assert_no_leakage(out)

    if verbose:
        print("train/val/heldout/matched:",
              len(out["train"]), len(out["val"]),
              len(out["heldout_test"]), len(out["decision_matched"]))
    return out


def assert_no_leakage(splits):
    """No validation or held-out text may appear in training in any variant."""
    for v in TEXT_VARIANTS_ALL:
        tr = set(splits["train"][v])
        assert tr.isdisjoint(set(splits["val"][v])), "train/val leak in " + v
        assert tr.isdisjoint(set(splits["heldout_test"][v])), "train/test leak in " + v


def assert_no_forbidden_inputs(text_col, batch_texts=None, notes=None):
    """Structural + contents check that labels never reach the model."""
    assert text_col in TEXT_VARIANTS_ALL or text_col == POOLED_COL, (
        "model input must be a text variant or the pooled column, "
        "got: {}".format(text_col))
    assert text_col not in FORBIDDEN_INPUT_COLS

    if batch_texts is not None and notes is not None:
        joined = "\n".join(batch_texts)
        for note in notes:
            probe = str(note)[:30].strip()
            if len(probe) >= 15:
                assert probe not in joined, (
                    "note text leaked into model input: {!r}".format(probe))


def subsample(df, n, seed):
    if n is None or len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)


def melt_variants(df):
    """Stack the four text variants into one pooled training/evaluation frame.

    scenario_family_id is retained so bootstrap confidence intervals resample
    semantic families rather than treating the four language realizations as
    independent observations.
    """
    keep = [
        "id", "scenario_family_id", "split", "in_decision_matched_test",
        "decision",
    ] + [c for c in SLICE_COLS if c in df.columns and c != "in_decision_matched_test"]

    # Preserve order without duplicates.
    keep = list(dict.fromkeys(keep))
    frames = []
    for v in TEXT_VARIANTS_ALL:
        sub = df[keep].copy()
        sub[POOLED_COL] = df[v].values
        sub["text_variant"] = v
        frames.append(sub)
    return pd.concat(frames, axis=0, ignore_index=True)


# ------------------------------------------------------------------
# Metrics
# ------------------------------------------------------------------
def _f(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else float("nan")
    except Exception:
        return float("nan")


def _safe(fn, *args, **kwargs):
    try:
        return _f(fn(*args, **kwargs))
    except Exception:
        return float("nan")


def softmax_np(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(probs, y_true, n_bins=N_CALIB_BINS):
    """Confidence-based ECE / MCE plus the reliability bins themselves."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, mce = 0.0, 0.0
    rows = []
    n = len(y_true)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        cnt = int(mask.sum())
        if cnt == 0:
            rows.append({"bin_lower": lo, "bin_upper": hi, "count": 0,
                         "avg_confidence": float("nan"), "accuracy": float("nan"),
                         "gap": float("nan")})
            continue
        avg_conf = float(conf[mask].mean())
        acc = float(correct[mask].mean())
        gap = abs(avg_conf - acc)
        ece += (cnt / n) * gap
        mce = max(mce, gap)
        rows.append({"bin_lower": lo, "bin_upper": hi, "count": cnt,
                     "avg_confidence": avg_conf, "accuracy": acc, "gap": gap})
    return float(ece), float(mce), pd.DataFrame(rows)


def fit_temperature(val_logits, val_y, max_iter=200):
    """Temperature scaling, fitted on VALIDATION ONLY.

    Never call this with gold/test data.
    """
    logits = torch.tensor(np.asarray(val_logits), dtype=torch.float32)
    labels = torch.tensor(np.asarray(val_y), dtype=torch.long)
    log_t = torch.zeros(1, requires_grad=True)
    optim = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter)
    nll = torch.nn.CrossEntropyLoss()

    def closure():
        optim.zero_grad()
        loss = nll(logits / torch.exp(log_t), labels)
        loss.backward()
        return loss

    try:
        optim.step(closure)
        t = float(torch.exp(log_t).item())
    except Exception:
        t = 1.0
    if not np.isfinite(t) or t <= 0:
        t = 1.0
    return float(np.clip(t, 0.05, 20.0))


def _binary_at_threshold(y_bin, score, thr):
    pred = (score >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        y_bin, pred, average="binary", zero_division=0)
    return _f(p), _f(r), _f(f)


def tune_binary_threshold(y_bin, score):
    """Best-F1 threshold. Tuned on validation, applied to test."""
    if len(np.unique(y_bin)) < 2:
        return 0.5, float("nan")
    prec, rec, thr = precision_recall_curve(y_bin, score)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros_like(prec), where=(prec + rec) > 0)
    # precision_recall_curve returns len(thr) == len(prec) - 1
    if len(thr) == 0:
        return 0.5, float("nan")
    best = int(np.nanargmax(f1[:-1]))
    return float(thr[best]), float(f1[best])


def _rate_at(curve_x, curve_y, target, mode):
    """mode='min_x_at_y' -> smallest x whose y >= target.
       mode='max_y_at_x' -> largest y whose x <= target."""
    curve_x = np.asarray(curve_x)
    curve_y = np.asarray(curve_y)
    if mode == "min_x_at_y":
        ok = curve_y >= target
        return float(curve_x[ok].min()) if ok.any() else float("nan")
    ok = curve_x <= target
    return float(curve_y[ok].max()) if ok.any() else float("nan")


def bootstrap_ci(y_true, y_pred, metric="f1_macro", n_boot=N_BOOTSTRAP,
                 alpha=0.05, seed=12345, groups=None):
    """95% percentile bootstrap CI.

    If `groups` is supplied, resampling occurs at semantic-family level. This
    keeps the four text variants (and any same-family rows) together and avoids
    artificially narrow confidence intervals.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n < 2:
        return float("nan"), float("nan")

    rng = np.random.default_rng(seed)
    fn = (lambda a, b: f1_score(a, b, average="macro",
                                labels=list(range(N_LABELS)), zero_division=0)
          ) if metric == "f1_macro" else (lambda a, b: accuracy_score(a, b))

    stats = np.empty(n_boot, dtype=np.float64)

    if groups is None:
        for i in range(n_boot):
            idx = rng.integers(0, n, n)
            try:
                stats[i] = fn(y_true[idx], y_pred[idx])
            except Exception:
                stats[i] = np.nan
    else:
        groups = np.asarray(groups)
        unique_groups = pd.unique(groups)
        group_to_idx = {g: np.flatnonzero(groups == g) for g in unique_groups}
        ng = len(unique_groups)
        for i in range(n_boot):
            sampled_groups = rng.choice(unique_groups, size=ng, replace=True)
            idx = np.concatenate([group_to_idx[g] for g in sampled_groups])
            try:
                stats[i] = fn(y_true[idx], y_pred[idx])
            except Exception:
                stats[i] = np.nan

    lo = float(np.nanpercentile(stats, 100 * alpha / 2))
    hi = float(np.nanpercentile(stats, 100 * (1 - alpha / 2)))
    return lo, hi


def compute_all_metrics(y_true, y_pred, probs, meta,
                        temperature=1.0, tuned_threshold=0.5,
                        do_bootstrap=True, bootstrap_seed=12345):
    """Every metric emitted for slice_type == 'overall'.

    meta : DataFrame aligned with y_true holding the slice columns.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    probs = np.asarray(probs, dtype=np.float64)
    all_ids = list(range(N_LABELS))
    m = {}

    # ---- classification ----
    m["accuracy"] = _safe(accuracy_score, y_true, y_pred)
    m["balanced_accuracy"] = _safe(balanced_accuracy_score, y_true, y_pred)
    for avg in ["macro", "micro", "weighted"]:
        m["f1_" + avg] = _safe(f1_score, y_true, y_pred, average=avg,
                               labels=all_ids, zero_division=0)
    for avg in ["macro", "weighted"]:
        m["precision_" + avg] = _safe(precision_score, y_true, y_pred, average=avg,
                                      labels=all_ids, zero_division=0)
        m["recall_" + avg] = _safe(recall_score, y_true, y_pred, average=avg,
                                   labels=all_ids, zero_division=0)
    m["cohen_kappa"] = _safe(cohen_kappa_score, y_true, y_pred, labels=all_ids)
    m["mcc"] = _safe(matthews_corrcoef, y_true, y_pred)

    # ---- per class ----
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=all_ids, zero_division=0)
    for i, lab in enumerate(LABELS):
        m["precision_" + lab] = _f(p[i])
        m["recall_" + lab] = _f(r[i])
        m["f1_" + lab] = _f(f[i])
        m["support_" + lab] = _f(s[i])

    # ---- binary collapse: ALLOW vs (REVIEW + BLOCK) ----
    # positive class = "needs attention", score = 1 - p(ALLOW)
    y_bin = (y_true != LABEL2ID["ALLOW"]).astype(int)
    score = 1.0 - probs[:, LABEL2ID["ALLOW"]]
    both_classes = len(np.unique(y_bin)) == 2

    m["bin_auprc"] = _safe(average_precision_score, y_bin, score) if both_classes else float("nan")
    m["bin_roc_auc"] = _safe(roc_auc_score, y_bin, score) if both_classes else float("nan")

    if both_classes:
        fpr, tpr, _ = roc_curve(y_bin, score)
        m["bin_fpr_at_tpr95"] = _rate_at(fpr, tpr, 0.95, "min_x_at_y")
        m["bin_fpr_at_tpr99"] = _rate_at(fpr, tpr, 0.99, "min_x_at_y")
        m["bin_tpr_at_fpr1"] = _rate_at(fpr, tpr, 0.01, "max_y_at_x")
        m["bin_tpr_at_fpr5"] = _rate_at(fpr, tpr, 0.05, "max_y_at_x")
    else:
        for k in ["bin_fpr_at_tpr95", "bin_fpr_at_tpr99",
                  "bin_tpr_at_fpr1", "bin_tpr_at_fpr5"]:
            m[k] = float("nan")

    bp, br, bf = precision_recall_fscore_support(
        y_bin, (y_pred != LABEL2ID["ALLOW"]).astype(int),
        average="binary", zero_division=0)[:3]
    m["bin_precision_argmax"], m["bin_recall_argmax"], m["bin_f1_argmax"] = _f(bp), _f(br), _f(bf)

    tp, tr_, tf = _binary_at_threshold(y_bin, score, tuned_threshold)
    m["bin_precision_tuned"], m["bin_recall_tuned"], m["bin_f1_tuned"] = tp, tr_, tf
    m["bin_best_f1_threshold"] = _f(tuned_threshold)

    # ---- calibration ----
    ece, mce, _bins = expected_calibration_error(probs, y_true)
    m["ece"] = ece
    m["mce"] = mce
    onehot = np.eye(N_LABELS)[y_true]
    m["brier_multiclass"] = _f(np.mean(np.sum((probs - onehot) ** 2, axis=1)))
    m["brier_binary"] = _f(np.mean((score - y_bin) ** 2))
    m["nll"] = _safe(log_loss, y_true, probs, labels=all_ids)
    m["temperature"] = _f(temperature)
    m["ece_before_temp"] = ece
    m["ece_after_temp"] = ece  # overwritten by the caller when raw logits exist

    # ---- safety ----
    hn = meta["hard_negative_type"].values
    si = meta["security_intent"].values

    benign = (hn != "not_applicable") & (y_true == LABEL2ID["ALLOW"])
    m["over_refusal_n"] = _f(benign.sum())
    m["over_refusal_rate"] = _f(np.mean(y_pred[benign] != LABEL2ID["ALLOW"])) if benign.any() else float("nan")
    m["over_refusal_block_rate"] = _f(np.mean(y_pred[benign] == LABEL2ID["BLOCK"])) if benign.any() else float("nan")

    oss = hn == "operational_self_service"
    m["over_escalation_n"] = _f(oss.sum())
    m["over_escalation_rate"] = _f(np.mean(y_pred[oss] == LABEL2ID["BLOCK"])) if oss.any() else float("nan")

    attack = si != "B0_legitimate_security"
    m["attack_n"] = _f(attack.sum())
    m["attack_recall"] = _f(np.mean(y_pred[attack] != LABEL2ID["ALLOW"])) if attack.any() else float("nan")

    is_block = y_true == LABEL2ID["BLOCK"]
    m["block_n"] = _f(is_block.sum())
    m["catastrophic_miss_rate"] = _f(np.mean(y_pred[is_block] == LABEL2ID["ALLOW"])) if is_block.any() else float("nan")
    m["block_to_review_rate"] = _f(np.mean(y_pred[is_block] == LABEL2ID["REVIEW"])) if is_block.any() else float("nan")

    is_review = y_true == LABEL2ID["REVIEW"]
    m["review_n"] = _f(is_review.sum())
    m["review_to_allow_rate"] = _f(np.mean(y_pred[is_review] == LABEL2ID["ALLOW"])) if is_review.any() else float("nan")
    m["review_to_block_rate"] = _f(np.mean(y_pred[is_review] == LABEL2ID["BLOCK"])) if is_review.any() else float("nan")

    # ---- uncertainty ----
    if do_bootstrap:
        groups = (meta["scenario_family_id"].values
                  if "scenario_family_id" in meta.columns else None)
        lo, hi = bootstrap_ci(
            y_true, y_pred, "f1_macro", seed=bootstrap_seed, groups=groups)
        m["f1_macro_ci_low"], m["f1_macro_ci_high"] = lo, hi
        lo, hi = bootstrap_ci(
            y_true, y_pred, "accuracy", seed=bootstrap_seed, groups=groups)
        m["accuracy_ci_low"], m["accuracy_ci_high"] = lo, hi
    else:
        for k in ["f1_macro_ci_low", "f1_macro_ci_high",
                  "accuracy_ci_low", "accuracy_ci_high"]:
            m[k] = float("nan")

    assert set(m) == set(METRIC_KEYS), (
        "METRIC_KEYS drift: missing={} extra={}".format(
            sorted(set(METRIC_KEYS) - set(m)), sorted(set(m) - set(METRIC_KEYS)))
    )
    return m


def confusion_matrices(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_LABELS)))
    with np.errstate(invalid="ignore", divide="ignore"):
        cmn = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)
    return cm, np.nan_to_num(cmn, nan=0.0)


def compute_slices(y_true, y_pred, probs, meta, text_variant):
    """Per-slice rows. Flag low support, never drop it."""
    rows = []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    def emit(slice_type, slice_value, mask):
        n = int(mask.sum())
        if n == 0:
            rows.append((slice_type, slice_value, "f1_macro", float("nan"), 0, True))
            rows.append((slice_type, slice_value, "accuracy", float("nan"), 0, True))
            rows.append((slice_type, slice_value, "recall_not_allow", float("nan"), 0, True))
            return
        yt, yp = y_true[mask], y_pred[mask]
        low = n < MIN_SLICE_SUPPORT
        rows.append((slice_type, slice_value, "f1_macro",
                     _safe(f1_score, yt, yp, average="macro",
                           labels=list(range(N_LABELS)), zero_division=0), n, low))
        rows.append((slice_type, slice_value, "accuracy",
                     _safe(accuracy_score, yt, yp), n, low))
        # "caught" rate -- for attack slices this is attack recall
        rows.append((slice_type, slice_value, "recall_not_allow",
                     _f(np.mean(yp != LABEL2ID["ALLOW"])), n, low))

    for col in SLICE_COLS:
        vals = meta[col].values
        for v in sorted(pd.unique(vals)):
            emit(col, str(v), vals == v)

    # Per-variant breakdown of the pooled model. When the eval frame is melted
    # it carries a real `text_variant` column, so we report macro-F1 for each
    # variant separately (bangla / banglish / code_mix / english). Falls back
    # to the scalar tag for a non-melted frame.
    if "text_variant" in meta.columns:
        tv = meta["text_variant"].values
        for v in sorted(pd.unique(tv)):
            emit("text_variant", str(v), tv == v)
    else:
        emit("text_variant", text_variant, np.ones(len(y_true), dtype=bool))
    return rows


# ------------------------------------------------------------------
# Artifacts
# ------------------------------------------------------------------
PRODUCED_FILES = set()


def _register(out_dir, path):
    PRODUCED_FILES.add(os.path.relpath(path, out_dir).replace(os.sep, "/"))


def run_key(experiment, variant, eval_set, seed):
    return "{}_{}_{}_seed{}".format(experiment, variant, eval_set, seed)


def dump_predictions(out_dir, model_slug, experiment, seed, eval_set,
                     text_variant, ids, y_true, y_pred, probs, meta):
    source_variant = (
        meta["text_variant"].astype(str).values
        if "text_variant" in meta.columns
        else np.array([text_variant] * len(y_true))
    )
    df = pd.DataFrame({
        "model_slug": model_slug,
        "experiment": experiment,
        "seed": seed,
        "eval_set": eval_set,
        "text_variant": text_variant,
        "source_text_variant": source_variant,
        "id": list(ids),
        "scenario_family_id": meta["scenario_family_id"].values,
        "split": meta["split"].values,
        "in_decision_matched_test": meta["in_decision_matched_test"].values,
        "y_true": [ID2LABEL[int(i)] for i in y_true],
        "y_pred": [ID2LABEL[int(i)] for i in y_pred],
        "p_ALLOW": probs[:, LABEL2ID["ALLOW"]],
        "p_REVIEW": probs[:, LABEL2ID["REVIEW"]],
        "p_BLOCK": probs[:, LABEL2ID["BLOCK"]],
        "domain": meta["domain"].values,
        "security_intent": meta["security_intent"].values,
        "risk_level": meta["risk_level"].values,
        "hard_negative_type": meta["hard_negative_type"].values,
    })
    assert list(df.columns) == PRED_COLUMNS, "PRED_COLUMNS drift"
    path = os.path.join(
        out_dir, "predictions",
        run_key(experiment, text_variant, eval_set, seed) + ".csv"
    )
    df.to_csv(path, index=False)
    _register(out_dir, path)
    return path, df


def dump_curves(out_dir, experiment, variant, eval_set, seed,
                y_true, probs, reliability_df):
    key = run_key(experiment, variant, eval_set, seed)
    y_bin = (np.asarray(y_true) != LABEL2ID["ALLOW"]).astype(int)
    score = 1.0 - np.asarray(probs)[:, LABEL2ID["ALLOW"]]

    if len(np.unique(y_bin)) == 2:
        fpr, tpr, roc_thr = roc_curve(y_bin, score)
        prec, rec, pr_thr = precision_recall_curve(y_bin, score)
        pr_thr = np.append(pr_thr, np.nan)
        roc_df = pd.DataFrame({"curve": "roc", "x_fpr": fpr, "y_tpr": tpr,
                               "threshold": roc_thr})
        pr_df = pd.DataFrame({"curve": "pr", "x_recall": rec, "y_precision": prec,
                              "threshold": pr_thr})
        curves = pd.concat([roc_df, pr_df], ignore_index=True)
    else:
        curves = pd.DataFrame(columns=["curve", "x_fpr", "y_tpr", "threshold",
                                       "x_recall", "y_precision"])

    p1 = os.path.join(out_dir, "curves", key + "_pr_roc.csv")
    curves.to_csv(p1, index=False)
    _register(out_dir, p1)

    p2 = os.path.join(out_dir, "curves", key + "_reliability.csv")
    reliability_df.to_csv(p2, index=False)
    _register(out_dir, p2)
    return p1, p2


def make_latex_table(df, caption, label, float_fmt="{:.3f}"):
    """Minimal booktabs table. No external dependency."""
    cols = list(df.columns)
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{" + "l" * 1 + "r" * (len(cols) - 1) + "}",
        r"\toprule",
        " & ".join(str(c).replace("_", r"\_") for c in cols) + r" \\",
        r"\midrule",
    ]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, float):
                cells.append("--" if not np.isfinite(v) else float_fmt.format(v))
            else:
                cells.append(str(v).replace("_", r"\_"))
        lines.append(" & ".join(cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{tab:" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def build_artifact_manifest(run_plan, figure_seed, save_checkpoints):
    """Every path the run is contractually required to produce."""
    manifest = {"run_meta.json", "results.json", "metrics_long.csv"}
    for name in ["main_results", "per_class", "per_slice", "efficiency"]:
        manifest.add("tables/{}.tex".format(name))

    for r in run_plan:
        exp, var, seed = r["experiment"], r["variant"], r["seed"]
        manifest.add("training/{}_{}_seed{}_history.csv".format(exp, var, seed))
        # Checkpoints are written for FIGURE_SEED only -> one saved model per
        # experiment, not one per seed.
        if save_checkpoints and seed == figure_seed:
            manifest.add("training/{}_{}_seed{}_best_checkpoint".format(exp, var, seed))
        for es in r["eval_sets"]:
            key = run_key(exp, var, es, seed)
            manifest.add("predictions/{}.csv".format(key))
            manifest.add("curves/{}_pr_roc.csv".format(key))
            manifest.add("curves/{}_reliability.csv".format(key))
            if seed == figure_seed:
                for fig in ["confusion", "reliability", "pr_curve",
                            "roc_curve", "per_class_f1"]:
                    manifest.add("figures/{}_{}.png".format(key, fig))
        if seed == figure_seed:
            manifest.add("figures/{}_{}_seed{}_training_curves.png".format(exp, var, seed))
    return manifest


def write_artifacts(out_dir, run_meta, results, metrics_rows, tables):
    p = os.path.join(out_dir, "run_meta.json")
    with open(p, "w") as fh:
        json.dump(run_meta, fh, indent=2, default=str)
    _register(out_dir, p)

    p = os.path.join(out_dir, "results.json")
    with open(p, "w") as fh:
        json.dump(results, fh, indent=2, default=str)
    _register(out_dir, p)

    df = pd.DataFrame(metrics_rows, columns=METRICS_LONG_COLUMNS)
    assert list(df.columns) == METRICS_LONG_COLUMNS, "METRICS_LONG_COLUMNS drift"
    p = os.path.join(out_dir, "metrics_long.csv")
    df.to_csv(p, index=False)
    _register(out_dir, p)

    for name, tex in tables.items():
        p = os.path.join(out_dir, "tables", "{}.tex".format(name))
        with open(p, "w") as fh:
            fh.write(tex)
        _register(out_dir, p)
    return df


def scan_produced(out_dir):
    """What is actually on disk, relative to out_dir.

    Hidden OS/tooling files (.DS_Store, .ipynb_checkpoints, ...) are not
    contract artifacts and are ignored so the manifest check is stable across
    macOS / editors / Kaggle.
    """
    found = set()
    for root, dirs, files in os.walk(out_dir):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for d in list(dirs):
            if d.endswith("_best_checkpoint"):
                found.add(os.path.relpath(os.path.join(root, d), out_dir).replace(os.sep, "/"))
                dirs.remove(d)
        for f in files:
            if f.startswith("."):
                continue
            found.add(os.path.relpath(os.path.join(root, f), out_dir).replace(os.sep, "/"))
    return found


def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("COMMON BLOCK v{} loaded".format(COMMON_VERSION))
# ===== END COMMON BLOCK v1.0.1 =====


## Cell 3 — Preflight

Resolves the `splits_v2` input, verifies the derived CSVs against the cleaned master, checks text/family leakage, reports split distributions, and inspects XLM-R-base before training.


In [ ]:
# ============================================================
# CELL 3 — PREFLIGHT (splits_v2)
# ============================================================

import transformers

DATA_DIR = resolve_data_dir()
OUT_DIR = resolve_out_dir(MODEL_SLUG)
DEVICE_PROFILE = resolve_device_profile()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MASTER_CSV = os.path.join(DATA_DIR, MASTER_FILENAME)
TRAIN_CSV = os.path.join(DATA_DIR, DERIVED_FILES["train"])
VAL_CSV = os.path.join(DATA_DIR, DERIVED_FILES["val"])
HELDOUT_CSV = os.path.join(DATA_DIR, DERIVED_FILES["heldout"])
MATCHED_CSV = os.path.join(DATA_DIR, DERIVED_FILES["matched"])

for p in [MASTER_CSV, TRAIN_CSV, VAL_CSV, HELDOUT_CSV, MATCHED_CSV]:
    assert os.path.exists(p), p

print("\nMASTER_CSV :", MASTER_CSV)
print("TRAIN_CSV  :", TRAIN_CSV)
print("VAL_CSV    :", VAL_CSV)
print("HELDOUT_CSV:", HELDOUT_CSV)
print("MATCHED_CSV:", MATCHED_CSV)

# ------------------------------------------------------------------
# Release consistency checks
# ------------------------------------------------------------------
_master = pd.read_csv(MASTER_CSV)
_train_file = pd.read_csv(TRAIN_CSV)
_val_file = pd.read_csv(VAL_CSV)
_heldout_file = pd.read_csv(HELDOUT_CSV)
_matched_file = pd.read_csv(MATCHED_CSV)

required_cols = [
    "id", "domain", "security_intent", "hard_negative_type", "decision",
    "risk_level", "code_mix", "banglish", "bangla", "english",
    "scenario_family_id", "split", "in_decision_matched_test",
]
_missing = [c for c in required_cols if c not in _master.columns]
assert not _missing, "master missing columns: {}".format(_missing)
assert _master.id.is_unique
assert set(_master.split) == {"train", "val", "heldout_test"}
assert set(_master.decision) <= set(LABELS)

assert set(_train_file.id) == set(_master.loc[_master.split == "train", "id"])
assert set(_val_file.id) == set(_master.loc[_master.split == "val", "id"])
assert set(_heldout_file.id) == set(
    _master.loc[_master.split == "heldout_test", "id"]
)
assert set(_matched_file.id) == set(
    _master.loc[
        (_master.split == "heldout_test")
        & (_master.in_decision_matched_test.astype(bool)),
        "id"
    ]
)

print("\nrelease counts:")
print("  master          :", len(_master))
print("  train           :", len(_train_file))
print("  val             :", len(_val_file))
print("  heldout_test    :", len(_heldout_file))
print("  decision_matched:", len(_matched_file))

print("\nby domain / split:")
print(pd.crosstab(_master.domain, _master.split, margins=True))

print("\nheldout decision distribution:")
print(pd.crosstab(_heldout_file.domain, _heldout_file.decision, margins=True))

print("\ndecision-matched distribution:")
print(pd.crosstab(_matched_file.domain, _matched_file.decision, margins=True))

# ------------------------------------------------------------------
# Leakage checks: exact text + semantic family
# ------------------------------------------------------------------
for _v in TEXT_VARIANTS_ALL:
    _tr = set(_train_file[_v])
    assert _tr.isdisjoint(set(_val_file[_v])), "train/val leak in " + _v
    assert _tr.isdisjoint(set(_heldout_file[_v])), "train/heldout leak in " + _v

assert set(_train_file.scenario_family_id).isdisjoint(
    set(_val_file.scenario_family_id))
assert set(_train_file.scenario_family_id).isdisjoint(
    set(_heldout_file.scenario_family_id))
assert set(_val_file.scenario_family_id).isdisjoint(
    set(_heldout_file.scenario_family_id))

print("\nleakage audit: PASS (all four text variants + scenario_family_id)")

print("\nlabel distribution:")
print(pd.crosstab(_master.decision, _master.split, margins=True))

print("\nlibrary versions:")
print("  transformers", transformers.__version__)
print("  torch       ", torch.__version__)

# ------------------------------------------------------------------
# XLM-R-base model/tokenizer preflight
# ------------------------------------------------------------------
from transformers import AutoConfig, AutoTokenizer

_cfg = AutoConfig.from_pretrained(MODEL_NAME)
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nmodel preflight:")
print("  architecture :", _cfg.architectures)
print("  model_type   :", _cfg.model_type)
print("  vocab_size   :", _cfg.vocab_size)
print("  hidden_size  :", _cfg.hidden_size)
print("  n_layers     :", getattr(_cfg, "num_hidden_layers", None))
print("  num_labels   :", N_LABELS)
print("  tokenizer    :", type(_tok).__name__)
print("  pad_token    :", _tok.pad_token, _tok.pad_token_id)
print("  padding_side :", _tok.padding_side)
print("  model_max_len:", _tok.model_max_length)

_probe = _tok(
    ["ব্যাংক অ্যাকাউন্ট ভেরিফাই করতে চাই", "transfer all funds now"],
    padding=True, truncation=True, max_length=MAX_LEN
)
print("  probe ids    :", [len(x) for x in _probe["input_ids"]])


## Cell 4 — Experiment / data preparation


In [ ]:
# ============================================================
# CELL 4 — EXPERIMENT / DATA PREPARATION
# ============================================================

if MIN_SLICE_SUPPORT_OVERRIDE is not None:
    MIN_SLICE_SUPPORT = MIN_SLICE_SUPPORT_OVERRIDE

ACTIVE_SEEDS = SMOKE_SEEDS if SMOKE_RUN else SEEDS

if RUN_FULL_EXPERIMENTS or SMOKE_RUN:
    ACTIVE_EXPERIMENTS = list(EXPERIMENTS)
else:
    _match = [
        k for k, v in EXPERIMENTS.items()
        if v["test_domain"] == TEST_DOMAIN
        and set(v["train_domains"]) == set(TRAIN_DOMAINS)
    ]
    assert _match, "Cell 1 TRAIN_DOMAINS/TEST_DOMAIN match no known experiment"
    ACTIVE_EXPERIMENTS = _match

print("active experiments:", ACTIVE_EXPERIMENTS)
print("active variants   :", TEXT_VARIANTS)
print("active seeds      :", ACTIVE_SEEDS)

SPLIT_CACHE = {}
for _exp in ACTIVE_EXPERIMENTS:
    _spec = EXPERIMENTS[_exp]
    SPLIT_CACHE[_exp] = load_splits(
        train_domains=_spec["train_domains"],
        test_domain=_spec["test_domain"],
        master_csv=MASTER_CSV,
    )
    _s = SPLIT_CACHE[_exp]
    print(
        "{:16s} train={:5d} val={:4d} heldout={:4d} matched={:3d}".format(
            _exp, len(_s["train"]), len(_s["val"]),
            len(_s["heldout_test"]), len(_s["decision_matched"])
        )
    )

DROPPED_IDS = []  # v2 release is already cleaned before publication

def eval_sets_for(experiment):
    return ["val", "heldout_test", "decision_matched"]

for _v in TEXT_VARIANTS:
    assert_no_forbidden_inputs(_v)
print("\ninput-feature contract: only text variants reach the tokenizer")
print("forbidden columns     :", FORBIDDEN_INPUT_COLS)

ALL_CONFIGS = []
for _exp in ACTIVE_EXPERIMENTS:
    for _seed in ACTIVE_SEEDS:
        ALL_CONFIGS.append({
            "experiment": _exp,
            "variant": POOLED_TAG,
            "seed": _seed,
            "eval_sets": eval_sets_for(_exp),
        })

RUN_PLAN = [
    c for i, c in enumerate(ALL_CONFIGS)
    if i % SHARD_COUNT == SHARD_INDEX
]

print("\nrun plan: {} configurations (shard {}/{} of {})".format(
    len(RUN_PLAN), SHARD_INDEX, SHARD_COUNT, len(ALL_CONFIGS)))

if not SMOKE_RUN and RUN_FULL_EXPERIMENTS and SHARD_COUNT == 1:
    assert set(c["experiment"] for c in RUN_PLAN) == EXPECTED_EXPERIMENTS
    assert set(c["variant"] for c in RUN_PLAN) == {POOLED_TAG}
    assert set(c["seed"] for c in RUN_PLAN) == set(SEEDS)
    assert len(RUN_PLAN) == len(EXPERIMENTS) * len(SEEDS)
    print("coverage check: 4 experiments x 3 seeds = 12 pooled-model runs OK")


## Cell 5 — XLM-R-base model


In [ ]:
# ============================================================
# CELL 5 — MODEL
# ============================================================

from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_KIND = "encoder"

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the released strings exactly as provided. This avoids environment-
# dependent normalization and keeps Bangla/Banglish/code-mix/English treatment
# consistent across runs.
PREPROCESSING = "identity"


def preprocess_text(text):
    return str(text)


USE_AMP = bool(USE_MIXED_PRECISION and torch.cuda.is_available())
AMP_DTYPE = (
    torch.bfloat16
    if USE_AMP and DEVICE_PROFILE.get("is_ampere_plus", False)
    else torch.float16
)
SCALER_ENABLED = bool(USE_AMP and AMP_DTYPE == torch.float16)

print("preprocessing       :", PREPROCESSING)
print("mixed precision     :", USE_AMP)
print("AMP dtype           :", AMP_DTYPE if USE_AMP else "disabled")
print("GradScaler enabled  :", SCALER_ENABLED)


def build_model(seed):
    set_all_seeds(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=N_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    if GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    model.to(DEVICE)
    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, 100.0 * trainable / max(total, 1)


def trainable_state_dict(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def save_best_checkpoint(model, path):
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    TOKENIZER.save_pretrained(path)


_m = build_model(42)
_t, _tr, _pct = count_params(_m)
print("total params     : {:,}".format(_t))
print("trainable params : {:,} ({:.2f}%)".format(_tr, _pct))
del _m
free_memory()

USE_4BIT = False


## Cell 6 — Training


In [ ]:
# ============================================================
# CELL 6 — TRAINING
# ============================================================

from contextlib import nullcontext
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup


def amp_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()


def make_grad_scaler():
    if not SCALER_ENABLED:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


class GuardDataset(Dataset):
    """Only the selected text column is tokenized."""

    def __init__(self, df, text_col, tokenizer, max_len):
        assert_no_forbidden_inputs(text_col)
        self.texts = [preprocess_text(t) for t in df[text_col].tolist()]
        self.labels = [LABEL2ID[d] for d in df["decision"].tolist()]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": self.labels[i]}


def make_collate(tokenizer, max_len):
    def collate(batch):
        enc = tokenizer(
            [b["text"] for b in batch],
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(
            [b["label"] for b in batch], dtype=torch.long
        )
        return enc
    return collate


def class_weights_from(df):
    counts = np.array(
        [(df["decision"] == lab).sum() for lab in LABELS],
        dtype=np.float64,
    )
    counts = np.where(counts == 0, 1.0, counts)
    w = counts.sum() / (N_LABELS * counts)
    return torch.tensor(w, dtype=torch.float32)


@torch.no_grad()
def predict_logits(model, df, text_col, max_len, batch_size):
    model.eval()
    ds = GuardDataset(df, text_col, TOKENIZER, max_len)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate(TOKENIZER, max_len),
    )
    out = []
    for batch in dl:
        batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_context():
            logits = model(**batch).logits
        out.append(logits.float().cpu().numpy())
    return np.concatenate(out, axis=0)


def train_one(model, train_df, val_df, text_col, seed, cfg_name):
    """Train and return (best_state, history, stats)."""
    set_all_seeds(seed)

    epochs = SMOKE_EPOCHS if SMOKE_RUN else EPOCHS
    max_steps = SMOKE_MAX_STEPS if SMOKE_RUN else None

    ds = GuardDataset(train_df, text_col, TOKENIZER, MAX_LEN)
    dl = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=True,
        drop_last=False,
        collate_fn=make_collate(TOKENIZER, MAX_LEN),
    )

    steps_per_epoch = max(1, math.ceil(len(dl) / GRAD_ACCUM))
    total_steps = max_steps if max_steps else steps_per_epoch * epochs

    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if ("bias" in n or "norm" in n.lower()) else decay).append(p)

    optim = torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=LR,
    )
    sched = get_linear_schedule_with_warmup(
        optim,
        int(WARMUP_RATIO * total_steps),
        total_steps,
    )

    weights = class_weights_from(train_df)
    scaler = make_grad_scaler()

    history = []
    best_f1, best_state, best_epoch = -1.0, None, -1
    epochs_since_improve = 0
    early_stopped = False
    step = 0
    stop = False
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        running, nb = 0.0, 0
        optim.zero_grad(set_to_none=True)

        for i, batch in enumerate(dl):
            labels = batch.pop("labels")
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            with amp_context():
                logits = model(**batch).logits
                loss = torch.nn.functional.cross_entropy(
                    logits.float(),
                    labels.to(logits.device),
                    weight=weights.to(logits.device),
                ) / GRAD_ACCUM

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running += float(loss.detach().item()) * GRAD_ACCUM
            nb += 1

            if (i + 1) % GRAD_ACCUM == 0 or (i + 1) == len(dl):
                if scaler is not None:
                    scaler.unscale_(optim)

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    MAX_GRAD_NORM,
                )

                if scaler is not None:
                    scaler.step(optim)
                    scaler.update()
                else:
                    optim.step()

                sched.step()
                optim.zero_grad(set_to_none=True)
                step += 1

                if max_steps and step >= max_steps:
                    stop = True
                    break

        # ---- validation ----
        val_logits = predict_logits(
            model, val_df, text_col, MAX_LEN, EVAL_BATCH
        )
        val_true = np.array([LABEL2ID[d] for d in val_df["decision"]])
        val_pred = val_logits.argmax(axis=1)
        val_f1 = float(
            f1_score(
                val_true,
                val_pred,
                average="macro",
                labels=list(range(N_LABELS)),
                zero_division=0,
            )
        )
        val_acc = float(accuracy_score(val_true, val_pred))

        history.append(
            {
                "epoch": epoch,
                "step": step,
                "train_loss": running / max(nb, 1),
                "val_macro_f1": val_f1,
                "val_accuracy": val_acc,
                "lr": sched.get_last_lr()[0],
                "elapsed_s": time.time() - t0,
            }
        )
        print(
            "    epoch {} step {:4d} loss {:.4f} val_macroF1 {:.4f}".format(
                epoch, step, running / max(nb, 1), val_f1
            )
        )

        if val_f1 > best_f1 + EARLY_STOP_MIN_DELTA:
            best_f1, best_epoch = val_f1, epoch
            best_state = trainable_state_dict(model)
            epochs_since_improve = 0
        else:
            if val_f1 > best_f1:
                best_f1, best_epoch = val_f1, epoch
                best_state = trainable_state_dict(model)
            epochs_since_improve += 1

        if stop:
            break

        if not SMOKE_RUN and epochs_since_improve >= PATIENCE:
            early_stopped = True
            print(
                "    early stop: no val macro-F1 gain > {:.1e} for {} epochs "
                "(best epoch {}, best F1 {:.4f})".format(
                    EARLY_STOP_MIN_DELTA,
                    PATIENCE,
                    best_epoch,
                    best_f1,
                )
            )
            break

    if best_state is None:
        best_state = trainable_state_dict(model)
        best_epoch = 0

    train_time = time.time() - t0
    peak_mem = (
        torch.cuda.max_memory_allocated() / 1024 ** 3
        if torch.cuda.is_available()
        else 0.0
    )
    ckpt_mb = (
        sum(v.numel() * v.element_size() for v in best_state.values())
        / 1024 ** 2
    )

    return best_state, pd.DataFrame(history), {
        "best_val_macro_f1": best_f1,
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "early_stopped": bool(early_stopped),
        "train_time_s": train_time,
        "peak_gpu_mem_gb": float(peak_mem),
        "checkpoint_size_mb": float(ckpt_mb),
        "total_steps": step,
    }


def load_best(model, best_state):
    model.load_state_dict(best_state, strict=False)
    return model


@torch.no_grad()
def measure_latency(model, sample_texts, batch_sizes=(1, 16), n_rep=8):
    """Inference latency and throughput."""
    model.eval()
    out = {}

    for bs in batch_sizes:
        texts = (
            sample_texts
            * ((bs // max(len(sample_texts), 1)) + 1)
        )[:bs]
        enc = TOKENIZER(
            [preprocess_text(t) for t in texts],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        ).to(DEVICE)

        for _ in range(2):
            with amp_context():
                model(**enc)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.time()
        for _ in range(n_rep):
            with amp_context():
                model(**enc)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        dt = (time.time() - t0) / n_rep
        out["latency_bs{}_ms".format(bs)] = dt * 1000.0
        out["throughput_bs{}_per_s".format(bs)] = (
            bs / dt if dt > 0 else float("nan")
        )

    return out


print("training utilities ready")


## Cell 7 — Evaluation and run orchestration


In [ ]:
# ============================================================
# CELL 7 — EVALUATION  (+ run orchestration)
# ============================================================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def _fig_path(experiment, variant, eval_set, seed, name):
    return os.path.join(OUT_DIR, "figures",
                        "{}_{}.png".format(run_key(experiment, variant, eval_set, seed), name))


def save_figures(experiment, variant, eval_set, seed, y_true, y_pred, probs,
                 reliability_df, metrics):
    cm, cmn = confusion_matrices(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_LABELS)); ax.set_xticklabels(LABELS, rotation=45, ha="right")
    ax.set_yticks(range(N_LABELS)); ax.set_yticklabels(LABELS)
    for i in range(N_LABELS):
        for j in range(N_LABELS):
            ax.text(j, i, "{}\n{:.2f}".format(cm[i, j], cmn[i, j]),
                    ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title("{} / {} / {}".format(experiment, variant, eval_set), fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    p = _fig_path(experiment, variant, eval_set, seed, "confusion")
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)

    fig, ax = plt.subplots(figsize=(4.0, 3.8))
    rel = reliability_df.dropna(subset=["avg_confidence"])
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.plot(rel["avg_confidence"], rel["accuracy"], "o-", lw=1.5)
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_title("reliability (ECE={:.3f})".format(metrics["ece"]), fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    fig.tight_layout()
    p = _fig_path(experiment, variant, eval_set, seed, "reliability")
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)

    y_bin = (np.asarray(y_true) != LABEL2ID["ALLOW"]).astype(int)
    score = 1.0 - np.asarray(probs)[:, LABEL2ID["ALLOW"]]
    has_both = len(np.unique(y_bin)) == 2

    fig, ax = plt.subplots(figsize=(4.0, 3.8))
    if has_both:
        prec, rec, _ = precision_recall_curve(y_bin, score)
        ax.plot(rec, prec, lw=1.6)
    ax.set_xlabel("recall"); ax.set_ylabel("precision")
    ax.set_title("PR: ALLOW vs REVIEW+BLOCK (AUPRC={:.3f})".format(
        metrics["bin_auprc"]), fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    fig.tight_layout()
    p = _fig_path(experiment, variant, eval_set, seed, "pr_curve")
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)

    fig, ax = plt.subplots(figsize=(4.0, 3.8))
    if has_both:
        fpr, tpr, _ = roc_curve(y_bin, score)
        ax.plot(fpr, tpr, lw=1.6)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC (AUC={:.3f})".format(metrics["bin_roc_auc"]), fontsize=9)
    fig.tight_layout()
    p = _fig_path(experiment, variant, eval_set, seed, "roc_curve")
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)

    fig, ax = plt.subplots(figsize=(4.0, 3.2))
    ax.bar(LABELS, [metrics["f1_" + l] for l in LABELS], color="#4C78A8")
    ax.set_ylim(0, 1); ax.set_ylabel("F1")
    ax.set_title("per-class F1", fontsize=9)
    fig.tight_layout()
    p = _fig_path(experiment, variant, eval_set, seed, "per_class_f1")
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)


def save_training_curves(experiment, variant, seed, history):
    fig, ax1 = plt.subplots(figsize=(4.6, 3.4))
    ax1.plot(history["epoch"], history["train_loss"], "o-", color="#E45756",
             label="train loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="#E45756")
    ax2 = ax1.twinx()
    ax2.plot(history["epoch"], history["val_macro_f1"], "s-", color="#4C78A8",
             label="val macro-F1")
    ax2.set_ylabel("val macro-F1", color="#4C78A8"); ax2.set_ylim(0, 1)
    ax1.set_title("{} / {} / seed {}".format(experiment, variant, seed), fontsize=9)
    fig.tight_layout()
    p = os.path.join(OUT_DIR, "figures",
                     "{}_{}_seed{}_training_curves.png".format(experiment, variant, seed))
    fig.savefig(p, dpi=130); plt.close(fig); _register(OUT_DIR, p)


def evaluate_split(model, df, text_col, experiment, variant, eval_set, seed,
                   temperature, threshold, do_bootstrap):
    """Predict, score, dump predictions/curves/figures. Returns metrics."""
    logits = predict_logits(model, df, text_col, MAX_LEN, EVAL_BATCH)
    y_true = np.array([LABEL2ID[d] for d in df["decision"]])
    probs_raw = softmax_np(logits, 1.0)
    probs = softmax_np(logits, temperature)
    y_pred = logits.argmax(axis=1)

    # Carry text_variant through (present on melted/pooled frames) so
    # compute_slices can report per-variant macro-F1 for the unified model.
    meta_cols = [
        "domain", "security_intent", "risk_level", "hard_negative_type",
        "scenario_family_id", "split", "in_decision_matched_test",
    ]
    if "text_variant" in df.columns:
        meta_cols = meta_cols + ["text_variant"]
    meta = df[meta_cols]

    metrics = compute_all_metrics(
        y_true, y_pred, probs_raw, meta,
        temperature=temperature, tuned_threshold=threshold,
        do_bootstrap=do_bootstrap, bootstrap_seed=seed)

    ece_before, _, rel_before = expected_calibration_error(probs_raw, y_true)
    ece_after, _, rel_after = expected_calibration_error(probs, y_true)
    metrics["ece_before_temp"] = ece_before
    metrics["ece_after_temp"] = ece_after

    dump_predictions(OUT_DIR, MODEL_SLUG, experiment, seed, eval_set, variant,
                     df["id"].tolist(), y_true, y_pred, probs_raw, meta)
    dump_curves(OUT_DIR, experiment, variant, eval_set, seed,
                y_true, probs_raw, rel_before)

    if seed == FIGURE_SEED:
        save_figures(experiment, variant, eval_set, seed,
                     y_true, y_pred, probs_raw, rel_before, metrics)

    slice_rows = compute_slices(y_true, y_pred, probs_raw, meta, variant)
    return metrics, slice_rows, len(df)


def run_all():
    results = {"model_slug": MODEL_SLUG, "common_version": COMMON_VERSION,
               "experiments": {}}
    metrics_rows = []
    efficiency_rows = []

    for n_cfg, cfg in enumerate(RUN_PLAN, 1):
        exp, var, seed = cfg["experiment"], cfg["variant"], cfg["seed"]
        tag = "{} / {} / seed {}".format(exp, var, seed)
        print("\n[{}/{}] {}".format(n_cfg, len(RUN_PLAN), tag))

        splits = SPLIT_CACHE[exp]

        # Unified model: stack all 4 variants into POOLED_COL. Melt first, then
        # subsample in smoke so the smoke set still spans every variant.
        train_df = melt_variants(splits["train"])
        val_df = melt_variants(splits["val"])
        eval_dfs = {
            "val": val_df,
            "heldout_test": melt_variants(splits["heldout_test"]),
            "decision_matched": melt_variants(splits["decision_matched"]),
        }

        if SMOKE_RUN:
            train_df = subsample(train_df, SMOKE_TRAIN_N, seed)
            val_df = subsample(val_df, SMOKE_VAL_N, seed)
            eval_dfs = {
                "val": val_df,
                "heldout_test": subsample(
                    eval_dfs["heldout_test"], SMOKE_TEST_N, seed),
                "decision_matched": subsample(
                    eval_dfs["decision_matched"], SMOKE_TEST_N, seed),
            }

        # --- resume support ---
        done = all(os.path.exists(os.path.join(
            OUT_DIR, "predictions", run_key(exp, var, es, seed) + ".csv"))
            for es in cfg["eval_sets"])
        if done and not FORCE_RETRAIN:
            print("    already complete, skipping (FORCE_RETRAIN=False)")
            for es in cfg["eval_sets"]:
                _register(OUT_DIR, os.path.join(
                    OUT_DIR, "predictions", run_key(exp, var, es, seed) + ".csv"))
            continue

        # --- contract: only POOLED_COL is tokenized, and it never carries a
        # forbidden column's text. melt_variants drops `note` entirely; the
        # label/slice columns remain as dataframe fields but are never fed to
        # the tokenizer (GuardDataset reads POOLED_COL only). ---
        _probe = GuardDataset(train_df.head(8), POOLED_COL, TOKENIZER, MAX_LEN)
        assert_no_forbidden_inputs(POOLED_COL)
        assert "note" not in train_df.columns, (
            "pooled frame must not carry the free-text `note` column")

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        model = build_model(seed)
        total_p, train_p, pct_p = count_params(model)

        best_state, history, train_stats = train_one(
            model, train_df, val_df, POOLED_COL, seed, tag)
        load_best(model, best_state)

        hp = os.path.join(OUT_DIR, "training",
                          "{}_{}_seed{}_history.csv".format(exp, var, seed))
        history.to_csv(hp, index=False)
        _register(OUT_DIR, hp)

        if seed == FIGURE_SEED:
            save_training_curves(exp, var, seed, history)

        # Save the deployable checkpoint for FIGURE_SEED only -> one model per
        # experiment (4 per notebook), not one per seed.
        if SAVE_CHECKPOINTS and seed == FIGURE_SEED:
            cp = os.path.join(OUT_DIR, "training",
                              "{}_{}_seed{}_best_checkpoint".format(exp, var, seed))
            save_best_checkpoint(model, cp)
            _register(OUT_DIR, cp)

        # --- calibration + threshold: VALIDATION ONLY ---
        val_logits = predict_logits(model, val_df, POOLED_COL, MAX_LEN, EVAL_BATCH)
        val_true = np.array([LABEL2ID[d] for d in val_df["decision"]])
        temperature = fit_temperature(val_logits, val_true)
        val_score = 1.0 - softmax_np(val_logits, 1.0)[:, LABEL2ID["ALLOW"]]
        threshold, _ = tune_binary_threshold(
            (val_true != LABEL2ID["ALLOW"]).astype(int), val_score)
        print("    temperature={:.3f}  tuned_threshold={:.3f}".format(
            temperature, threshold))

        lat = measure_latency(model, eval_dfs["heldout_test"][POOLED_COL].tolist()[:32])

        exp_node = results["experiments"].setdefault(exp, {})
        seed_node = exp_node.setdefault("seed_{}".format(seed), {})

        for es in cfg["eval_sets"]:
            metrics, slice_rows, n_rows = evaluate_split(
                model, eval_dfs[es], POOLED_COL, exp, var, es, seed,
                temperature, threshold,
                do_bootstrap=not SMOKE_RUN)

            seed_node.setdefault(es, {})[var] = {
                "overall": {"all": metrics}}

            for k, v in metrics.items():
                metrics_rows.append([MODEL_SLUG, exp, seed, es, var,
                                     "overall", "all", k, v, n_rows,
                                     n_rows < MIN_SLICE_SUPPORT])
            for st, sv, mk, mv, ns, low in slice_rows:
                metrics_rows.append([MODEL_SLUG, exp, seed, es, var,
                                     st, sv, mk, mv, ns, low])
                seed_node[es][var].setdefault(st, {}).setdefault(sv, {})[mk] = mv

            print("    {:10s} n={:5d} macroF1={:.4f} acc={:.4f}".format(
                es, n_rows, metrics["f1_macro"], metrics["accuracy"]))

        efficiency_rows.append(dict(
            model_slug=MODEL_SLUG, experiment=exp, text_variant=var, seed=seed,
            total_params=total_p, trainable_params=train_p,
            trainable_pct=pct_p, **train_stats, **lat))

        free_memory(model, best_state)

    return results, metrics_rows, pd.DataFrame(efficiency_rows)


RESULTS, METRICS_ROWS, EFFICIENCY_DF = run_all()
print("\nevaluation complete: {} metric rows".format(len(METRICS_ROWS)))


## Cell 8 — Emit artifacts

The smoke run is successful only when this cell reaches the final `PASSED` banner.


In [ ]:
EXTRA_RUN_META = {
    "preprocessing": PREPROCESSING,
    "mixed_precision": USE_AMP,
    "amp_dtype": str(AMP_DTYPE) if USE_AMP else "disabled",
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
}

# ============================================================
# CELL 8 — EMIT ARTIFACTS + CONTRACT ASSERTIONS
# ============================================================

import transformers

metrics_df = pd.DataFrame(METRICS_ROWS, columns=METRICS_LONG_COLUMNS)

# ------------------------------------------------------------------
# Tables
# ------------------------------------------------------------------
_overall = metrics_df[(metrics_df.slice_type == "overall")
                      & (metrics_df.metric == "f1_macro")]

main_tbl = (_overall.groupby(["experiment", "eval_set", "text_variant"])["value"]
            .agg(["mean", "std", "count"]).reset_index()
            .rename(columns={"mean": "macro_f1_mean", "std": "macro_f1_std",
                             "count": "n_seeds"}))

_per_class = metrics_df[(metrics_df.slice_type == "overall")
                        & (metrics_df.metric.isin(
                            ["f1_" + l for l in LABELS]))]
per_class_tbl = (_per_class.groupby(["experiment", "text_variant", "metric"])["value"]
                 .mean().reset_index())

_slices = metrics_df[(metrics_df.slice_type != "overall")
                     & (metrics_df.metric == "f1_macro")
                     & (~metrics_df.low_support)]
per_slice_tbl = (_slices.groupby(["slice_type", "slice_value"])["value"]
                 .agg(["mean", "std"]).reset_index())

eff_cols = [c for c in ["experiment", "text_variant", "seed", "trainable_params",
                        "trainable_pct", "train_time_s", "peak_gpu_mem_gb",
                        "latency_bs1_ms", "latency_bs16_ms",
                        "throughput_bs16_per_s", "checkpoint_size_mb"]
            if c in EFFICIENCY_DF.columns]
eff_tbl = EFFICIENCY_DF[eff_cols] if len(EFFICIENCY_DF) else pd.DataFrame(columns=eff_cols)

TABLES = {
    "main_results": make_latex_table(main_tbl, "Decision macro-F1 by experiment, eval set and text variant.", "main_results"),
    "per_class": make_latex_table(per_class_tbl, "Per-class F1.", "per_class"),
    "per_slice": make_latex_table(per_slice_tbl, "Sliced macro-F1 (low-support slices excluded).", "per_slice"),
    "efficiency": make_latex_table(eff_tbl, "Efficiency metrics.", "efficiency"),
}

# ------------------------------------------------------------------
# run_meta.json
# ------------------------------------------------------------------
_clean_counts = {}
for _e, _s in SPLIT_CACHE.items():
    _clean_counts[_e] = {k: int(len(_s[k]))
                         for k in ["train", "val", "heldout_test", "decision_matched"]}

RUN_META = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "model_kind": MODEL_KIND,
    "common_version": COMMON_VERSION,
    "smoke_run": SMOKE_RUN,
    "run_full_experiments": RUN_FULL_EXPERIMENTS,
    "experiments": ACTIVE_EXPERIMENTS,
    "text_variants": TEXT_VARIANTS,
    "seeds": ACTIVE_SEEDS,
    "figure_seed": FIGURE_SEED,
    "shard_index": SHARD_INDEX,
    "shard_count": SHARD_COUNT,
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        **DEVICE_PROFILE,
    },
    "data": {
        "data_dir": DATA_DIR,
        "master_csv": MASTER_CSV,
        "release": "splits_v2",
        "train_csv": TRAIN_CSV,
        "val_csv": VAL_CSV,
        "heldout_csv": HELDOUT_CSV,
        "decision_matched_csv": MATCHED_CSV,
        "dropped_duplicate_ids": DROPPED_IDS,
        "n_dropped_in_notebook": len(DROPPED_IDS),
        "split_counts": _clean_counts,
    },
    "config": {
        "max_len": MAX_LEN, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
        "grad_accum": GRAD_ACCUM, "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY, "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "aux_heads": AUX_HEADS, "min_slice_support": MIN_SLICE_SUPPORT,
        "save_checkpoints": SAVE_CHECKPOINTS,
    },
    "efficiency": (EFFICIENCY_DF.to_dict(orient="records")
                   if len(EFFICIENCY_DF) else []),
}
RUN_META.update(EXTRA_RUN_META)

metrics_df = write_artifacts(OUT_DIR, RUN_META, RESULTS, METRICS_ROWS, TABLES)

# ==================================================================
# CONTRACT ASSERTIONS
# ==================================================================
print("\n--- artifact contract ---")

# 1. manifest
MANIFEST = build_artifact_manifest(RUN_PLAN, FIGURE_SEED, SAVE_CHECKPOINTS)
produced = scan_produced(OUT_DIR)

missing = MANIFEST - produced
extra = produced - MANIFEST
if missing:
    print("MISSING ({}):".format(len(missing)))
    for m in sorted(missing)[:20]:
        print("   ", m)
if extra:
    print("EXTRA ({}):".format(len(extra)))
    for m in sorted(extra)[:20]:
        print("   ", m)
assert not missing and not extra, "ARTIFACT_MANIFEST mismatch"
print("1. ARTIFACT_MANIFEST  : {} paths, exact match".format(len(MANIFEST)))

# 2. METRIC_KEYS
_overall_keys = set(metrics_df[metrics_df.slice_type == "overall"].metric)
assert _overall_keys == set(METRIC_KEYS), (
    "METRIC_KEYS drift: {}".format(_overall_keys.symmetric_difference(set(METRIC_KEYS))))
print("2. METRIC_KEYS        : {} keys, exact match".format(len(METRIC_KEYS)))

# 3. PRED_COLUMNS
_pred_files = sorted(os.listdir(os.path.join(OUT_DIR, "predictions")))
for _f in _pred_files:
    _df = pd.read_csv(os.path.join(OUT_DIR, "predictions", _f))
    assert list(_df.columns) == PRED_COLUMNS, "PRED_COLUMNS drift in " + _f
print("3. PRED_COLUMNS       : {} files, exact order".format(len(_pred_files)))

# 4. artifact round-trip -- predictions alone must regenerate macro-F1
_checked = 0
for _f in _pred_files:
    _df = pd.read_csv(os.path.join(OUT_DIR, "predictions", _f))
    _exp, _seed = _df.experiment.iloc[0], int(_df.seed.iloc[0])
    _es, _var = _df.eval_set.iloc[0], _df.text_variant.iloc[0]
    _recomputed = f1_score(
        [LABEL2ID[x] for x in _df.y_true],
        [LABEL2ID[x] for x in _df.y_pred],
        average="macro", labels=list(range(N_LABELS)), zero_division=0)
    _stored = metrics_df[(metrics_df.experiment == _exp)
                         & (metrics_df.seed == _seed)
                         & (metrics_df.eval_set == _es)
                         & (metrics_df.text_variant == _var)
                         & (metrics_df.slice_type == "overall")
                         & (metrics_df.metric == "f1_macro")].value
    if len(_stored) == 0:
        continue          # resumed config: predictions on disk, not rescored
    _stored = float(_stored.iloc[0])
    assert abs(_recomputed - _stored) < 1e-6, (
        "round-trip mismatch in {}: {} vs {}".format(_f, _recomputed, _stored))
    _checked += 1
print("4. artifact round-trip: {}/{} files, |delta| < 1e-6".format(
    _checked, len(_pred_files)))

# 5. common version
assert RUN_META["common_version"] == COMMON_VERSION
assert RESULTS["common_version"] == COMMON_VERSION
print("5. COMMON_VERSION     :", COMMON_VERSION)

# 6. leakage (re-asserted from the emitted predictions)
for _exp in ACTIVE_EXPERIMENTS:
    _s = SPLIT_CACHE[_exp]
    for _v in TEXT_VARIANTS:
        assert set(_s["train"][_v]).isdisjoint(set(_s["heldout_test"][_v]))
        assert set(_s["train"][_v]).isdisjoint(set(_s["val"][_v]))
    assert set(_s["train"].scenario_family_id).isdisjoint(
        set(_s["heldout_test"].scenario_family_id))
    assert set(_s["train"].scenario_family_id).isdisjoint(
        set(_s["val"].scenario_family_id))
    if EXPERIMENTS[_exp]["test_domain"]:
        assert EXPERIMENTS[_exp]["test_domain"] not in set(_s["train"].domain)
        assert EXPERIMENTS[_exp]["test_domain"] not in set(_s["val"].domain)
print("6. leakage            : no train/eval text or family overlap")

# 7. coverage
# Unified model: the text_variant COLUMN is always POOLED_TAG ("all"); the
# per-variant breakdown lives in text_variant SLICE rows.
_cov_exp = set(metrics_df.experiment)
_cov_var = set(metrics_df.text_variant)
_cov_seed = set(metrics_df.seed)
_cov_slice_var = set(
    metrics_df[metrics_df.slice_type == "text_variant"].slice_value)
print("7. coverage           : {} experiments, {} pooled-tags, {} seeds, "
      "{} variant-slices".format(
          len(_cov_exp), len(_cov_var), len(_cov_seed), len(_cov_slice_var)))
if SMOKE_RUN:
    assert _cov_exp == EXPECTED_EXPERIMENTS, _cov_exp
    assert _cov_var == {POOLED_TAG}, _cov_var
    assert _cov_slice_var == EXPECTED_VARIANTS, _cov_slice_var
    print("   smoke gate: all 4 experiments exercised as unified models, "
          "all 4 variants present as eval slices")

print("\n================================================")
print(" {}  {}  PASSED".format(
    MODEL_SLUG, "SMOKE" if SMOKE_RUN else "FULL"))
print(" output: {}".format(OUT_DIR))
print("================================================")


## Cell 9 — Package outputs for download


In [ ]:
# ============================================================
# CELL 9 — ZIP OUTPUTS FOR DOWNLOAD
# ============================================================
import shutil
_zip_base = os.path.join(os.path.dirname(OUT_DIR.rstrip("/")), "..", "{}_outputs".format(MODEL_SLUG))
_zip_base = os.path.abspath(_zip_base)
_zip_path = shutil.make_archive(_zip_base, "zip", root_dir=os.path.dirname(OUT_DIR.rstrip("/")),
                                base_dir=os.path.basename(OUT_DIR.rstrip("/")))
print("download this file:", _zip_path, "({:.1f} MB)".format(os.path.getsize(_zip_path) / 1e6))
